# JN13 — Parameter File: Builder & Validator
## A deep dive into the Tsunami-HySEA parameter file

**HySEALab · Preprocessing Notebooks · EDANYA Research Group, Universidad de Málaga**
*Edited by José Manuel González Vida*

---

### What this notebook is for

**This notebook is the reference for the Tsunami-HySEA parameter file**:
it explains the file line by line, then gives you two reusable tools.

| Goal | Tool in this notebook |
|------|-----------------------|
| **Understand** every line and the rules that govern it | Sections 1–3 (annotated tables) |
| **Build** a brand-new, correct parameter file from a pattern | `build_parfile(config)` — Section 4 |
| **Validate** an existing parameter file and get a diagnosis | `validate_parfile(path)` — Section 5 |

> The parameter file is **positional**: meaning is given by *line order*, not by keywords.
> A single missing or extra line shifts everything below it and silently corrupts the run.
> That is exactly why a builder and a validator are worth having.

---

### Requirements

**Python packages** (any recent version):

```bash
conda install -c conda-forge ipywidgets matplotlib netcdf4
# or: pip install ipywidgets matplotlib netCDF4
```

- `ipywidgets` — only for the interactive file picker (Section 0)
- `matplotlib` — only for the mesh-hierarchy diagram (Section 4-bis)
- `netCDF4` — only to infer submesh parents from grid extents (optional)

**Input data: none strictly required.** The notebook builds and validates its
own example files, so it runs end-to-end out of the box. To validate **your
own** parameter files, place them anywhere reachable (e.g. the `data/` folder
next to this notebook) and use the file picker in Section 0 or
`report("path/to/your_parfile.txt")`.

In [ ]:
import os, tempfile

# Optional reference parameter files — used in the demos below IF present.
# The notebook also generates its own examples, so these are not required.
DATA_DIR    = "data"
os.makedirs(DATA_DIR, exist_ok=True)
PARFILE_1L  = os.path.join(DATA_DIR, "parameters.txt")        # single grid
PARFILE_ND  = os.path.join(DATA_DIR, "parametersNested.txt")  # nested (2 levels)

print("Working directory:", os.getcwd())
for p in (PARFILE_1L, PARFILE_ND):
    status = 'FOUND  ' if os.path.exists(p) else 'MISSING (optional)'
    print(f"  {status} {p}")

---
## 0 — File picker (interactive shortcut)

If you just want to **validate a file** without editing any code, use the panel below:

- **Local** — the file lives on this server. A file browser lets you pick it; the format is
  checked **and** the paths it references (bathymetry, `POIs.dat`, nested grids) are verified
  on disk. For nested runs the **parent of each submesh** is inferred from the grid extents.
- **Remote** — you upload the file from your computer. Only the **format** and the nesting
  **structure** are checked; file paths are not verified and parent links are not drawn
  (the grids are not here).

> Run the whole notebook once (**Kernel ▸ Restart & Run All**) so that
> `validate_parfile()` / `report()` / `show_hierarchy()` are defined; then come back here
> and click **Validate**. Sections 1–5 explain the format and the tools in detail.


In [ ]:
# ── Graphical file picker for validation ────────────────────────────────────
# Two modes:
#   • Local  -> the file is on THIS server: format is validated AND the paths it
#               references (bathymetry, POIs, nested grids) are checked on disk;
#               for nested runs the parent of each submesh is inferred from the
#               grid extents.
#   • Remote -> you upload the file from your computer: only format and nesting
#               structure are checked (no paths, no parent links).
#
# NOTE: this panel only builds the widgets. Validation runs when you click
# "Validate", which calls report()/validate_parfile()/show_hierarchy()
# (defined in Sections 4-5). Run the whole notebook first (Restart & Run All).

import os, tempfile
import ipywidgets as widgets
from IPython.display import display, clear_output


class _FileBrowser:
    """Self-contained file browser. Navigation is driven by buttons (Home / Up /
    Open / a path box), which is robust; clicking a 📄 row selects the file.
    The previous version rebuilt the list inside the selection callback, which
    left ipywidgets in an inconsistent state and broke navigation."""

    def __init__(self, start=None):
        self.dir = os.path.abspath(start or os.getcwd())
        self.selected = None

        self._path = widgets.Text(value=self.dir, layout=widgets.Layout(width="520px"))
        self._home = widgets.Button(description="🏠 Home", layout=widgets.Layout(width="90px"))
        self._up   = widgets.Button(description="⬆ Up",    layout=widgets.Layout(width="70px"))
        self._go   = widgets.Button(description="Go",      layout=widgets.Layout(width="55px"))
        self._open = widgets.Button(description="Open folder ▸", button_style="info",
                                    tooltip="Enter the highlighted folder",
                                    layout=widgets.Layout(width="130px"))
        self._list = widgets.Select(rows=12, layout=widgets.Layout(width="640px"))
        self._info = widgets.HTML()

        self._home.on_click(lambda _: self._goto(os.path.expanduser("~")))
        self._up.on_click(lambda _: self._goto(os.path.dirname(self.dir)))
        self._go.on_click(lambda _: self._goto(self._path.value))
        self._open.on_click(self._open_selected)
        self._list.observe(self._on_select, "value")

        bar = widgets.HBox([self._home, self._up, self._path, self._go])
        self.widget = widgets.VBox([bar, self._list, self._open, self._info])
        self._refresh()

    def _entries(self):
        try:
            names = sorted(os.listdir(self.dir))
        except OSError as e:
            self._info.value = f"<span style='color:#b00'>Cannot open: {e}</span>"
            return []
        dirs  = [n for n in names if not n.startswith(".")
                 and os.path.isdir(os.path.join(self.dir, n))]
        files = [n for n in names if not n.startswith(".")
                 and os.path.isfile(os.path.join(self.dir, n))]
        return ([(f"📁 {n}", ("DIR", n)) for n in dirs] +
                [(f"📄 {n}", ("FILE", n)) for n in files])

    def _show_info(self):
        sel = (f"  •  selected: <b>{os.path.basename(self.selected)}</b>"
               if self.selected else "")
        self._info.value = f"Folder: <code>{self.dir}</code>{sel}"

    def _refresh(self):
        # Rebuild the listing OUTSIDE the value-callback (called only from buttons).
        self._path.value = self.dir
        self._list.unobserve(self._on_select, "value")
        self._list.options = self._entries()
        self._list.value = None
        self._list.observe(self._on_select, "value")
        self._show_info()

    def _goto(self, path):
        path = os.path.abspath(os.path.expanduser((path or "").strip())) if path else self.dir
        if os.path.isdir(path):
            self.dir, self.selected = path, None
            self._refresh()
        else:
            self._info.value = f"<span style='color:#b00'>Not a folder: {path}</span>"

    def _open_selected(self, _):
        v = self._list.value
        if isinstance(v, tuple) and v[0] == "DIR":
            self._goto(os.path.join(self.dir, v[1]))
        else:
            self._info.value = ("<span style='color:#b00'>Highlight a 📁 folder, "
                                "then click “Open folder ▸”.</span>")

    def _on_select(self, change):
        # Only file selection happens here — no list rebuild, so it is safe.
        v = change["new"]
        if isinstance(v, tuple) and v[0] == "FILE":
            self.selected = os.path.join(self.dir, v[1])
            self._show_info()


_mode = widgets.RadioButtons(
    options=[
        ("Local — on this server (also checks paths to grids/POIs)", "local"),
        ("Remote — upload a file (format and structure only)",       "remote"),
    ],
    description="Source:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="auto"),
)

_local_picker = _FileBrowser(os.getcwd())
_local_widget = _local_picker.widget

_uploader = widgets.FileUpload(accept=".txt,.dat", multiple=False,
                               description="Upload file")

_tree_chk = widgets.Checkbox(value=True, indent=False,
                             description="Show nested-mesh hierarchy")
_validate_btn = widgets.Button(description="Validate", button_style="primary", icon="check")
_out = widgets.Output()

_local_box  = widgets.VBox([widgets.HTML("<b>Local file</b> — browse and click a 📄 file"),
                            _local_widget])
_remote_box = widgets.VBox([widgets.HTML("<b>Remote file</b>"), _uploader])

def _refresh_mode(*_):
    _local_box.layout.display  = "" if _mode.value == "local"  else "none"
    _remote_box.layout.display = "" if _mode.value == "remote" else "none"
_mode.observe(_refresh_mode, "value")
_refresh_mode()

def _uploaded_file():
    """Return (name, bytes) of the uploaded file; ipywidgets 7/8 compatible."""
    val = _uploader.value
    if not val:
        return None
    if isinstance(val, (list, tuple)):          # ipywidgets >= 8
        item = val[0]
        name, content = item.get("name", "uploaded.txt"), item["content"]
    else:                                       # ipywidgets 7.x (keyed by name)
        name = next(iter(val))
        content = val[name]["content"]
    data = content.tobytes() if hasattr(content, "tobytes") else bytes(content)
    return name, data

def _validate_and_show(path, check_files, base_dir=None):
    report(path, check_files=check_files, base_dir=base_dir)
    if _tree_chk.value and "show_hierarchy" in globals():
        print()
        show_hierarchy(path, resolve_grids=check_files, base_dir=base_dir)

def _on_validate(_):
    with _out:
        clear_output()
        if "report" not in globals():
            print("⚠ Run the whole notebook first (Kernel ▸ Restart & Run All)\n"
                  "  so that validate_parfile()/report() are defined.")
            return
        if _mode.value == "local":
            path = _local_picker.selected
            if not path or not os.path.exists(path):
                print("⚠ Select an existing local file (click a 📄 row in the browser).")
                return
            _validate_and_show(path, check_files=True)
        else:
            up = _uploaded_file()
            if up is None:
                print("⚠ Upload a file first.")
                return
            name, data = up
            with tempfile.NamedTemporaryFile("wb", suffix="_" + name, delete=False) as tf:
                tf.write(data)
                tmp_path = tf.name
            try:
                print(f"(remote file: {name} — format and structure only)\n")
                _validate_and_show(tmp_path, check_files=False)
            finally:
                os.unlink(tmp_path)

_validate_btn.on_click(_on_validate)

display(widgets.VBox([_mode, _local_box, _remote_box,
                      _tree_chk, _validate_btn, _out]))

---
## 1 — How the file is organised

The parameter file is a **plain-text, line-oriented** file. Rules of the format:

1. **One value (or one space-separated vector) per line.** The line number *is* the meaning.
2. **Everything after `#` is a comment** and is ignored by the solver. Comments are optional
   but strongly recommended — they are the only documentation the file carries.
3. **Blank lines and pure-comment lines are ignored**, so you may add spacing freely.
4. **Order is fixed.** You cannot reorder lines. Some blocks are *conditional* (Section 3) —
   their presence depends on a flag written earlier in the file.

The overall skeleton, top to bottom:

```
HEADER        name, bathymetry, source (Okada), output prefix, variables to save
LEVELS        number of grids; for nested runs, one block per refinement level
DOMAIN        boundary conditions, total time, saving time
TIME SERIES   optional block — only present if "read points from file" = 1
NUMERICS      CFL, thresholds, stability, friction, scaling
```

The ten output flags (line *"variables to save"*) are, **in this exact order**:

| # | Variable | # | Variable |
|---|----------|---|----------|
| 1 | `eta` (free surface) | 6 | `maximum_modulus_of_velocity` |
| 2 | `maximum_eta` | 7 | `maximum_modulus_of_mass_flow` |
| 3 | `velocities` (u, v) | 8 | `momentum_flux` |
| 4 | `maximum_velocities` | 9 | `maximum_momentum_flux` |
| 5 | `modulus_of_velocity` | 10 | `arrival_times` |


---
## 2 — The single-grid file, line by line

This is an example single-grid file (`data/parameters.txt`). Every line annotated:

| # | Line value | Parameter | Meaning / valid values |
|---|------------|-----------|------------------------|
| 1 | `MEDITERRANEAN` | Experiment name | free label, appears in output |
| 2 | `EM_450m_topobathy.grd` | Bathymetry file | NetCDF `.grd` (x, y, z); **negative = ocean** (GMT) |
| 3 | `1` | Initialization mode | `1` = standard Okada; other modes read deformation from file |
| 4 | `0` | Kajiura filter | `0` off · `1` on (smooths the seafloor signal in deep water) |
| 5 | `1` | Number of faults *N* | integer ≥ 1 — defines how many Okada lines follow |
| 6 | `0.0 14.99 37.15 9.8 85.8 23.0 67.5 50 90 4.0` | Okada fault | `time lon lat depth length width strike dip rake slip` (×*N* lines) |
| 7 | `0` | Okada window | `0` = deform whole domain; `1` = crop to a window (coords follow) |
| 8 | `simulation_EM_propagation` | Output prefix | NetCDF output file name (no extension) |
| 9 | `1 1 0 0 0 0 0 0 0 1` | Variables to save | 10 flags (see table in Section 1) |
| 10 | `1` | Number of levels | `1` = single grid (no nesting) |
| 11–14 | `1 / 1 / 1 / 1` | Boundary conditions | upper / lower / left / right — `1` open, `-1` wall |
| 15 | `14400.1` | Total simulation time | seconds |
| 16 | `600.0` | NetCDF saving time | seconds (`-1` = do not save fields) |
| 17 | `1` | Read points from file? | `0` no · `1` yes → **enables the time-series block** |
| 18 | `POIs.dat` | Points Of Interest file | *only present if line 17 = 1* |
| 19 | `60.0` | Time-series saving time | seconds — *only present if line 17 = 1* |
| 20 | `0.5` | CFL | Courant number, `0 < CFL ≤ 1` (≈ 0.5 typical) |
| 21 | `5e-3` | epsilon_h | wet/dry threshold (m) |
| 22 | `20.0` | WAF threshold | depth (m) above which the 2s+WAF scheme is used |
| 23 | `0.2` | Stability coefficient | one value (single grid); **one per level** if nested |
| 24 | `0` | Friction type | `0` = Manning |
| 25 | `0.02` | Manning coefficient | bottom friction |
| 26 | `100.0` | Max water velocity | clipping value (m/s) |
| 27 | `1000000.0` | Typical length *L* | non-dimensionalisation scale |
| 28 | `1000.0` | Typical height *H* | non-dimensionalisation scale |
| 29 | `1e-3` | Arrival-time threshold | min `eta` (m) counted as "wave arrived" |


---
## 3 — The four rules that trip people up

These are the *conditional* parts of the format — where the number of lines depends on a value
written earlier. Get one wrong and every line below shifts.

**Rule 1 — The fault block scales with *N*.**
Line *"number of faults"* = `N` ⇒ exactly `N` Okada lines follow, one per sub-fault.
For a finite-fault / multi-segment source, set `N` and stack the lines.

**Rule 2 — The time-series block is optional.**
If *"read points from file"* = `1`, **two** lines follow: the POIs filename *and* the time-series
saving interval. If it is `0`, **neither** line exists — the next line is the CFL.
This is the single most common cause of a "shifted" file.

**Rule 3 — The stability coefficient counts levels.**
Single grid → one number (`0.2`). Nested run with *L* levels → *L* numbers on one line
(`0.2 0.2 0.2 0.2` for 4 levels).

**Rule 4 — The level block repeats and nests submeshes.**
For each level `i ≥ 1`: a *refinement ratio*, a *number of submeshes* `M`, then `M` triplets of
(grid file, output prefix, save flags). See Section 3-bis.


### 3-bis — The nested (multi-grid) layout

When *number of levels* > 1, an extra block is inserted **right after the level-0 save flags**,
before the boundary conditions. An example 2-level file (`data/parametersNested.txt`):

| # | Line value | Meaning |
|---|------------|---------|
| 8 | `simulation_..._L0` | level-0 output prefix |
| 9 | `0 0 0 0 0 0 0 0 0 0` | level-0 save flags |
| 10 | `2` | **number of levels** (here 2) |
| 11 | `4` | refinement ratio of level 1 (cells are 4× finer) |
| 12 | `1` | number of submeshes in level 1 |
| 13 | `EM_112m_bathy.grd` | grid file, level-1 submesh 1 |
| 14 | `simulation_..._L1` | output prefix, level-1 submesh 1 |
| 15 | `1 1 0 0 0 0 0 0 0 1` | save flags, level-1 submesh 1 |
| 16–19 | boundary conditions | … the file continues as the single-grid version |

The repeating unit, for **each level `i` from 1 to L-1**:

```
<refinement ratio of level i>
<number of submeshes M_i>
   repeat M_i times:
       <grid file>
       <output prefix>
       <save flags>
```

A 4-level domain with two coastal patches per level (e.g. Eastern Sicily: Catania + Syracuse)
simply stacks three of these blocks. The builder below generates this automatically.


In [ ]:
# ── Canonical names of the 10 output flags (used by builder & validator) ─────
SAVE_VARS = [
    "eta", "maximum_eta", "velocities", "maximum_velocities",
    "modulus_of_velocity", "maximum_modulus_of_velocity",
    "maximum_modulus_of_mass_flow", "momentum_flux",
    "maximum_momentum_flux", "arrival_times",
]
N_SAVE = len(SAVE_VARS)   # 10

def _num(x):
    """Format a number compactly (avoids 1000000.0 -> 1e+06 surprises)."""
    if isinstance(x, float) and x == int(x) and abs(x) < 1e15:
        # keep a trailing .0 only where the format expects a real (times, scales)
        return repr(x)
    return f"{x:g}" if isinstance(x, float) else str(x)

print("Output flags, in order:")
for i, v in enumerate(SAVE_VARS, 1):
    print(f"  {i:2d}. {v}")

---
## 4 — Builder: write a correct file from a pattern

`build_parfile(config)` turns a structured Python `dict` into a valid parameter file.
It enforces every rule from Section 3 for you:

- emits exactly `len(config["faults"])` Okada lines,
- writes the time-series block **only** when a `pois_file` is given,
- repeats the stability coefficient once per level,
- expands the nested level/submesh blocks.

The `config` schema (single grid = one entry in `levels`; nested = more entries):

```python
config = {
    "name": "MEDITERRANEAN",
    "init_mode": 1, "kajiura": 0,
    "faults": [[time, lon, lat, depth, length, width, strike, dip, rake, slip], ...],
    "okada_window": 0,                     # 0, or [lon_min, lon_max, lat_min, lat_max]
    "levels": [                            # level 0 first
        {"bathymetry": "...grd", "output": "...", "save": [1,1,0,0,0,0,0,0,0,1]},
        # nested levels add "ratio" and a list of "submeshes":
        # {"ratio": 4, "submeshes": [{"bathymetry":..., "output":..., "save":[...]}]},
    ],
    "boundaries": {"upper": 1, "lower": 1, "left": 1, "right": 1},
    "sim_time": 14400.1, "nc_saving_time": 600.0,
    "pois_file": "POIs.dat", "ts_saving_time": 60.0,   # omit pois_file -> no TS block
    "cfl": 0.5, "epsilon_h": 5e-3, "waf_threshold": 20.0,
    "stability": 0.2,                      # scalar (repeated per level) or list
    "friction_type": 0, "manning": 0.02, "max_velocity": 100.0,
    "typical_length": 1e6, "typical_height": 1000.0, "arrival_threshold": 1e-3,
}
```


In [ ]:
def build_parfile(cfg):
    """Return the text of a Tsunami-HySEA parameter file from a config dict."""
    lines = []
    COL = 38  # column where inline comments start
    def add(value, comment):
        s = str(value)
        pad = " " * max(1, COL - len(s))
        lines.append(f"{s}{pad}# {comment}")

    levels = cfg["levels"]
    nlev   = len(levels)
    lv0    = levels[0]

    # ── Header ──────────────────────────────────────────────────────────────
    add(cfg["name"],                 "name of the experiment")
    add(lv0["bathymetry"],           "bathymetry file (level 0)")
    add(cfg.get("init_mode", 1),     "initialization mode (1: standard Okada)")
    add(cfg.get("kajiura", 0),       "apply Kajiura filter? (0: no, 1: yes)")

    # ── Source (Okada faults) ────────────────────────────────────────────────
    faults = cfg["faults"]
    add(len(faults),                 "number of faults")
    for k, f in enumerate(faults):
        cmt = ("Okada: time lon lat depth length width strike dip rake slip"
               if k == 0 else f"fault {k + 1}")
        add(" ".join(_num(v) for v in f), cmt)

    win = cfg.get("okada_window", 0)
    if not win:
        add(0, "Okada computation window? (0: whole domain)")
    else:
        add("1 " + " ".join(_num(v) for v in win),
            "Okada window: lon_min lon_max lat_min lat_max")

    # ── Level 0 output ────────────────────────────────────────────────────────
    add(lv0["output"],               "output filename prefix (level 0)")
    add(" ".join(str(int(v)) for v in lv0["save"]),
        "save flags: " + ", ".join(SAVE_VARS))

    # ── Levels / nesting ──────────────────────────────────────────────────────
    add(nlev,                        "number of levels")
    for i in range(1, nlev):
        lvl  = levels[i]
        subs = lvl["submeshes"]
        add(lvl["ratio"],            f"refinement ratio of level {i}")
        add(len(subs),               f"number of submeshes for level {i}")
        for j, s in enumerate(subs, 1):
            add(s["bathymetry"],     f"grid file (level {i}, submesh {j})")
            add(s["output"],         f"output prefix (level {i}, submesh {j})")
            add(" ".join(str(int(v)) for v in s["save"]), "save flags")

    # ── Domain / time ─────────────────────────────────────────────────────────
    b = cfg["boundaries"]
    add(b["upper"], "upper boundary condition (1: open, -1: wall)")
    add(b["lower"], "lower boundary condition (1: open, -1: wall)")
    add(b["left"],  "left boundary condition  (1: open, -1: wall)")
    add(b["right"], "right boundary condition (1: open, -1: wall)")
    add(_num(float(cfg["sim_time"])),       "total simulation time (s)")
    add(_num(float(cfg["nc_saving_time"])), "NetCDF saving time (s) (-1: do not save)")

    # ── Time-series block (conditional) ───────────────────────────────────────
    pois = cfg.get("pois_file")
    if pois:
        add(1, "read points from file? (1: yes)")
        add(pois, "file with the Points Of Interest")
        add(_num(float(cfg["ts_saving_time"])), "time-series saving time (s)")
    else:
        add(0, "read points from file? (0: no)")

    # ── Numerics ──────────────────────────────────────────────────────────────
    add(_num(cfg["cfl"]),           "CFL")
    add(_num(cfg["epsilon_h"]),     "epsilon_h (m)")
    add(_num(cfg["waf_threshold"]), "threshold for the 2s+WAF scheme (m)")

    stab = cfg["stability"]
    if isinstance(stab, (list, tuple)):
        stab_vals = list(stab)
    else:
        stab_vals = [stab] * nlev          # repeat scalar once per level
    add(" ".join(_num(v) for v in stab_vals),
        "stability coefficient" + (" (one per level)" if nlev > 1 else ""))

    add(cfg.get("friction_type", 0), "friction type (0: Manning)")
    add(_num(cfg["manning"]),        "Manning bottom friction")
    add(_num(float(cfg["max_velocity"])),    "maximum allowed water velocity (m/s)")
    add(_num(float(cfg["typical_length"])),  "typical length L")
    add(_num(float(cfg["typical_height"])),  "typical height H")
    add(_num(cfg["arrival_threshold"]),      "threshold for arrival times")

    return "\n".join(lines) + "\n"

print("build_parfile() defined.")

In [ ]:
# ── Pattern 1: a single-grid propagation run ────────────────────────────────
config_single = {
    "name": "MEDITERRANEAN",
    "init_mode": 1, "kajiura": 0,
    "faults": [
        # time  lon      lat      depth  length  width  strike  dip  rake  slip
        [0.0, 14.9887, 37.1526, 9.803, 85.845, 22.983, 67.5, 50.0, 90.0, 3.996],
    ],
    "okada_window": 0,
    "levels": [
        {"bathymetry": "EM_450m_topobathy.grd",
         "output": "simulation_EM_propagation",
         "save": [1, 1, 0, 0, 0, 0, 0, 0, 0, 1]},
    ],
    "boundaries": {"upper": 1, "lower": 1, "left": 1, "right": 1},
    "sim_time": 14400.1, "nc_saving_time": 600.0,
    "pois_file": "POIs.dat", "ts_saving_time": 60.0,
    "cfl": 0.5, "epsilon_h": 5e-3, "waf_threshold": 20.0,
    "stability": 0.2,
    "friction_type": 0, "manning": 0.02, "max_velocity": 100.0,
    "typical_length": 1e6, "typical_height": 1000.0, "arrival_threshold": 1e-3,
}

print(build_parfile(config_single))

In [ ]:
# ── Pattern 2: a 2-level nested run (one submesh) ───────────────────────────
config_nested = {
    "name": "MEDITERRANEAN_2LEVELS",
    "init_mode": 1, "kajiura": 0,
    "faults": [[0.0, 14.9887, 37.1526, 9.803, 85.845, 22.983, 67.5, 50.0, 90.0, 3.996]],
    "okada_window": 0,
    "levels": [
        {"bathymetry": "EM_450m_topobathy.grd",
         "output": "simulation_EM_propagation_L0",
         "save": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]},
        {"ratio": 4,
         "submeshes": [
             {"bathymetry": "EM_112m_bathy.grd",
              "output": "simulation_EM_propagation_L1",
              "save": [1, 1, 0, 0, 0, 0, 0, 0, 0, 1]},
         ]},
    ],
    "boundaries": {"upper": 1, "lower": 1, "left": 1, "right": 1},
    "sim_time": 14400.1, "nc_saving_time": 600.0,
    "pois_file": "POIs.dat", "ts_saving_time": 60.0,
    "cfl": 0.5, "epsilon_h": 5e-3, "waf_threshold": 20.0,
    "stability": 0.2,                      # scalar -> repeated once per level
    "friction_type": 0, "manning": 0.02, "max_velocity": 100.0,
    "typical_length": 1e6, "typical_height": 1000.0, "arrival_threshold": 1e-3,
}

text = build_parfile(config_nested)
print(text)

# Write it next to the reference file (uncomment to save):
# with open(os.path.join(DATA_DIR, "parametersNested_generated.txt"), "w") as fh:
#     fh.write(text)

In [ ]:
# ── Extract and display the nested-mesh hierarchy ───────────────────────────
# The parameter file is FLAT per level: it lists, for each level, its refinement
# ratio + a number of submeshes, but it does NOT record which parent submesh each
# child belongs to. Tsunami-HySEA resolves that by GEOMETRY: every child grid lies
# inside exactly one parent grid (the alignment rule). So:
#   • the structure (levels / submeshes) is read from the text alone — works for
#     remote files too;
#   • the parent of each submesh is inferred by reading the .grd extents, which is
#     only possible when the grids are on disk (local mode). In remote mode the
#     tree is shown grouped by level, without parent links.
# Assumes standard Okada (initialization mode 1), like the validator.

def extract_hierarchy(path):
    """Return the list of levels. Level 0 = base mesh; levels >=1 carry 'ratio'
    and 'submeshes' [{'grid','output','save'}, ...]. Pure text parsing."""
    rows = []
    with open(path, encoding="utf-8", errors="replace") as fh:
        for raw in fh:
            txt = raw.split("#", 1)[0].strip()
            if txt:
                rows.append(txt.split())

    i = 0
    def nxt():
        nonlocal i
        r = rows[i] if i < len(rows) else None
        i += 1
        return r
    def as_int(r, default=0):
        try:
            return int(float(r[0]))
        except (TypeError, ValueError, IndexError):
            return default
    def first(r, default="?"):
        return r[0] if r else default

    nxt()                          # experiment name
    base_grid = first(nxt())       # bathymetry (level 0)
    nxt(); nxt()                   # initialization mode, Kajiura filter
    nf = as_int(nxt())             # number of faults
    for _ in range(max(nf, 0)):    # Okada lines
        nxt()
    nxt()                          # Okada window flag (0, or "1 lon... lat...")
    base_out = first(nxt())        # output prefix (level 0)
    nxt()                          # save flags (level 0)
    nlev = as_int(nxt(), 1)        # number of levels

    levels = [{"level": 0, "ratio": None, "grid": base_grid, "output": base_out}]
    for lev in range(1, nlev):
        ratio = as_int(nxt(), None)
        nsub  = as_int(nxt(), 0)
        subs = []
        for _ in range(max(nsub, 0)):
            g = first(nxt()); o = first(nxt()); s = nxt()
            subs.append({"grid": g, "output": o, "save": s})
        levels.append({"level": lev, "ratio": ratio, "submeshes": subs})
    return levels


# ── Geometry helpers: read grid extents and infer parents ───────────────────
def _grid_bbox(filepath):
    """(xmin, xmax, ymin, ymax) of a .grd/.nc grid, or None if unreadable."""
    if not filepath or not os.path.exists(filepath):
        return None
    try:
        from netCDF4 import Dataset
    except Exception:
        return None
    try:
        ds = Dataset(filepath)
        names = list(ds.variables)
        xn = next((n for n in names if n in ("lon", "x", "longitude")), None)
        yn = next((n for n in names if n in ("lat", "y", "latitude")), None)
        if xn is None or yn is None:
            ds.close(); return None
        x = ds[xn][:]; y = ds[yn][:]; ds.close()
        return (float(x.min()), float(x.max()), float(y.min()), float(y.max()))
    except Exception:
        return None

def _bbox_area(b):
    return (b[1] - b[0]) * (b[3] - b[2])

def _bbox_center(b):
    return ((b[0] + b[1]) / 2.0, (b[2] + b[3]) / 2.0)

def _contains(parent, child, tol=1e-3):
    return (parent[0] - tol <= child[0] and child[1] <= parent[1] + tol and
            parent[2] - tol <= child[2] and child[3] <= parent[3] + tol)

def _pick_parent(child, parent_row):
    """Index of the parent node that owns `child`, by geographic containment."""
    cb = child.get("bbox")
    if cb is None:
        return None
    # 1) parents whose extent fully contains the child -> smallest such parent
    contain = [(i, p["bbox"]) for i, p in enumerate(parent_row)
               if p.get("bbox") and _contains(p["bbox"], cb)]
    if contain:
        return min(contain, key=lambda ip: _bbox_area(ip[1]))[0]
    # 2) parent whose extent contains the child centre
    cx, cy = _bbox_center(cb)
    for i, p in enumerate(parent_row):
        b = p.get("bbox")
        if b and b[0] <= cx <= b[1] and b[2] <= cy <= b[3]:
            return i
    # 3) last resort: nearest parent centre
    cand = [(i, p["bbox"]) for i, p in enumerate(parent_row) if p.get("bbox")]
    if not cand:
        return None
    return min(cand, key=lambda ip: (_bbox_center(ip[1])[0] - cx) ** 2 +
                                     (_bbox_center(ip[1])[1] - cy) ** 2)[0]


def _build_tree(path, base_dir=None, resolve_grids=True):
    """Return (levels, nodes_by_level, parents, geom_used).
    nodes_by_level[k] is the list of node dicts at mesh level k;
    parents[k][j] is the index into nodes_by_level[k-1] (or None)."""
    levels = extract_hierarchy(path)
    if base_dir is None:
        base_dir = os.path.dirname(os.path.abspath(path))

    nodes = [[{"tag": "Level 0", "grid": levels[0]["grid"],
               "output": levels[0]["output"]}]]
    for lvl in levels[1:]:
        nodes.append([{"tag": f"L{lvl['level']}  ×{lvl['ratio']}",
                       "grid": s["grid"], "output": s["output"]}
                      for s in lvl["submeshes"]])

    geom_used = False
    if resolve_grids:
        for row in nodes:
            for nd in row:
                nd["bbox"] = _grid_bbox(os.path.join(base_dir, nd["grid"]))
        geom_used = any(nd.get("bbox") for row in nodes for nd in row)

    parents = [[None]]
    for k in range(1, len(nodes)):
        parents.append([_pick_parent(c, nodes[k - 1]) if geom_used else None
                        for c in nodes[k]])
    return levels, nodes, parents, geom_used


def _draw_hierarchy(nodes, parents):
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    nlev, y_gap, x_gap = len(nodes), 1.7, 3.4
    maxw = max(len(r) for r in nodes)
    fig, ax = plt.subplots(figsize=(max(6, 1 + x_gap * maxw), 1.2 + 1.15 * nlev))
    ax.axis("off")
    pos = {}
    for k, row in enumerate(nodes):
        y, n = (nlev - 1 - k) * y_gap, len(row)
        for j, nd in enumerate(row):
            x = (j - (n - 1) / 2) * x_gap
            pos[(k, j)] = (x, y)
            ax.add_patch(mpatches.FancyBboxPatch(
                (x - 1.5, y - 0.45), 3.0, 0.9,
                boxstyle="round,pad=0.02", fc="#e8f0fe", ec="#3367d6", lw=1.4))
            ax.text(x, y + 0.14, nd["tag"], ha="center", va="center",
                    fontsize=9, fontweight="bold")
            ax.text(x, y - 0.18, os.path.basename(nd["grid"]),
                    ha="center", va="center", fontsize=7)

    edges_drawn = False
    for k in range(1, nlev):
        for j in range(len(nodes[k])):
            pj = parents[k][j]
            if pj is None:
                continue
            x2, y2 = pos[(k, j)]
            x1, y1 = pos[(k - 1, pj)]
            ax.plot([x1, x2], [y1 - 0.45, y2 + 0.45], color="#3367d6", lw=1.2)
            edges_drawn = True
    if not edges_drawn and nlev > 1:
        ax.text(0.5, 0.0,
                "Parent links not shown — the grid files are needed (local mode) "
                "to infer them from the mesh extents.",
                transform=ax.transAxes, ha="center", va="bottom",
                fontsize=8, color="gray")
    ax.autoscale()
    ax.margins(0.08)
    plt.show()


def show_hierarchy(path, draw=True, resolve_grids=True, base_dir=None):
    """Print the hierarchy as a text tree and, if draw=True, draw the diagram.
    With resolve_grids=True (local) the parent of each submesh is inferred from
    the grid extents; otherwise (remote) the tree is grouped by level only."""
    levels, nodes, parents, geom = _build_tree(path, base_dir=base_dir,
                                                resolve_grids=resolve_grids)
    print("Mesh hierarchy")
    print("=" * 60)
    print("Level 0  ·  base mesh")
    print(f"    grid : {nodes[0][0]['grid']}")
    print(f"    out  : {nodes[0][0]['output']}")
    cum = 1
    for k, lvl in enumerate(levels[1:], start=1):
        cum *= (lvl["ratio"] or 1)
        print(f"Level {lvl['level']}  ·  refinement ×{lvl['ratio']} "
              f"(×{cum} vs level 0)  ·  {len(lvl['submeshes'])} submesh(es)")
        for j, s in enumerate(lvl["submeshes"]):
            pj = parents[k][j]
            if pj is not None:
                ptxt = f"   ← parent: {os.path.basename(nodes[k - 1][pj]['grid'])}"
            elif geom:
                ptxt = "   ← parent: (not determined)"
            else:
                ptxt = ""
            print(f"    └─ submesh {j + 1}: {os.path.basename(s['grid'])}"
                  f"   →   {s['output']}{ptxt}")
    if len(levels) == 1:
        print("\n(single mesh — no nesting)")
        return levels
    if not geom and resolve_grids:
        print("\n(grid files not found on disk — parent links omitted)")
    elif not resolve_grids:
        print("\n(remote mode — parent links omitted; structure only)")
    if draw:
        try:
            _draw_hierarchy(nodes, parents)
        except Exception as e:
            print(f"\n(could not draw the diagram: {e})")
    return levels

print("extract_hierarchy() and show_hierarchy() defined.")

# Demo: use the reference nested file if available; otherwise show the
# hierarchy of the 2-level example generated with build_parfile() above
# (its grids are not on disk, so parent links are omitted).
if os.path.exists(PARFILE_ND):
    show_hierarchy(PARFILE_ND)
else:
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as _tf:
        _tf.write(build_parfile(config_nested))
        _demo_par = _tf.name
    try:
        show_hierarchy(_demo_par, resolve_grids=False)
    finally:
        os.unlink(_demo_par)

---
## 5 — Validator: diagnose an existing file

`validate_parfile(path)` parses a parameter file using the **same rules** the builder enforces,
following the conditional blocks exactly as the solver would. It reports:

- **ERRORS** — structural problems that will make the run fail or read garbage
  (missing lines, wrong token counts, non-numeric where a number is required, file ends early);
- **WARNINGS** — values that are legal but suspicious (CFL out of the usual range, a save-flag
  vector that is not 10 long, referenced grid file not found on disk, …).

It is deliberately tolerant of comments, blank lines and pure-comment continuation lines
(such as the multi-line Okada legend used in some distributed example files).


In [ ]:
def validate_parfile(path, base_dir=None, check_files=True):
    """Validate a Tsunami-HySEA parameter file. Returns (errors, warnings)."""
    errors, warnings = [], []
    if base_dir is None:
        base_dir = os.path.dirname(os.path.abspath(path))

    # ── Tokenize: drop comments, blank lines and pure-comment lines ──────────
    rows = []  # (original_line_no, text_without_comment, tokens)
    with open(path, encoding="utf-8", errors="replace") as fh:
        for n, raw in enumerate(fh, 1):
            txt = raw.split("#", 1)[0].strip()
            if txt:
                rows.append((n, txt, txt.split()))

    i = 0
    def take(name, optional=False):
        # optional=True: a missing line at the end is allowed (no error).
        nonlocal i
        if i >= len(rows):
            if not optional:
                errors.append(f"File ends before '{name}' — the file is incomplete.")
            return None
        r = rows[i]; i += 1
        return r
    def to_int(r, name):
        if r is None: return None
        try:
            return int(float(r[2][0]))
        except (ValueError, IndexError):
            errors.append(f"L{r[0]}: '{name}' must be an integer (got '{r[1]}').")
            return None
    def to_float(r, name):
        if r is None: return None
        try:
            return float(r[2][0])
        except (ValueError, IndexError):
            errors.append(f"L{r[0]}: '{name}' must be a number (got '{r[1]}').")
            return None
    def check_range(val, lo, hi, name, r, warn=True):
        if val is None: return
        if not (lo <= val <= hi):
            msg = f"L{r[0]}: '{name}' = {val} outside expected [{lo}, {hi}]."
            (warnings if warn else errors).append(msg)
    def check_save(r):
        if r is None: return
        flags = r[2]
        if any(f not in ("0", "1") for f in flags):
            errors.append(f"L{r[0]}: save flags must be 0/1 (got '{r[1]}').")
        if len(flags) != N_SAVE:
            warnings.append(f"L{r[0]}: expected {N_SAVE} save flags, found {len(flags)}.")
    def check_file(r, label):
        if r is None or not check_files: return
        fp = os.path.join(base_dir, r[1])
        if not os.path.exists(fp):
            warnings.append(f"L{r[0]}: {label} '{r[1]}' not found in {base_dir}.")

    # ── Header ────────────────────────────────────────────────────────────────
    take("experiment name")
    check_file(take("bathymetry file (level 0)"), "bathymetry")
    init = to_int(take("initialization mode"), "initialization mode")
    if init is not None and init != 1:
        warnings.append(f"Initialization mode = {init}; this validator assumes standard Okada (1).")
    kaj = to_int(take("Kajiura filter"), "Kajiura filter")
    if kaj not in (0, 1, None):
        errors.append(f"Kajiura filter must be 0 or 1 (got {kaj}).")

    # ── Faults ────────────────────────────────────────────────────────────────
    nf = to_int(take("number of faults"), "number of faults")
    if nf is not None and nf < 1:
        errors.append(f"Number of faults must be >= 1 (got {nf}).")
    for k in range(nf or 0):
        r = take(f"Okada line for fault {k + 1}")
        if r is None: break
        toks = r[2]
        if len(toks) < 10:
            errors.append(f"L{r[0]}: fault {k + 1} needs 10 values, found {len(toks)}.")
            continue
        try:
            t, lon, lat, dep, length, width, strike, dip, rake, slip = map(float, toks[:10])
            check_range(lat, -90, 90, "fault latitude", r)
            check_range(lon, -180, 360, "fault longitude", r)
            check_range(strike, 0, 360, "strike", r)
            check_range(dip, 0, 90, "dip", r)
            check_range(rake, -180, 180, "rake", r)
            if dep <= 0:    warnings.append(f"L{r[0]}: depth = {dep} should be > 0 (km).")
            if length <= 0: errors.append(f"L{r[0]}: fault length must be > 0.")
            if width <= 0:  errors.append(f"L{r[0]}: fault width must be > 0.")
            if slip <= 0:   warnings.append(f"L{r[0]}: slip = {slip} should be > 0 (m).")
        except ValueError:
            errors.append(f"L{r[0]}: fault {k + 1} contains non-numeric values.")

    # ── Okada window ──────────────────────────────────────────────────────────
    win = take("Okada computation window flag")
    if win is not None and win[2][0] not in ("0", "1"):
        warnings.append(f"L{win[0]}: Okada window flag is usually 0 or 1 (got '{win[2][0]}').")

    # ── Level 0 output ────────────────────────────────────────────────────────
    take("output prefix (level 0)")
    check_save(take("save flags (level 0)"))

    # ── Levels ────────────────────────────────────────────────────────────────
    nlev = to_int(take("number of levels"), "number of levels")
    if nlev is not None and nlev < 1:
        errors.append(f"Number of levels must be >= 1 (got {nlev}).")
    for lev in range(1, (nlev or 1)):
        ratio = to_int(take(f"refinement ratio (level {lev})"), f"refinement ratio (level {lev})")
        if ratio is not None and ratio < 2:
            warnings.append(f"Refinement ratio of level {lev} = {ratio} (usually >= 2).")
        nsub = to_int(take(f"number of submeshes (level {lev})"), f"number of submeshes (level {lev})")
        if nsub is not None and nsub < 1:
            errors.append(f"Number of submeshes of level {lev} must be >= 1 (got {nsub}).")
        for s in range(nsub or 0):
            check_file(take(f"grid file (level {lev}, submesh {s + 1})"), "nested grid")
            take(f"output prefix (level {lev}, submesh {s + 1})")
            check_save(take(f"save flags (level {lev}, submesh {s + 1})"))

    # ── Boundaries & time ─────────────────────────────────────────────────────
    for side in ("upper", "lower", "left", "right"):
        bc = to_int(take(f"{side} boundary condition"), f"{side} boundary condition")
        if bc not in (1, -1, None):
            errors.append(f"{side.capitalize()} boundary must be 1 (open) or -1 (wall), got {bc}.")
    st = to_float(take("total simulation time"), "total simulation time")
    if st is not None and st <= 0:
        errors.append(f"Total simulation time must be > 0 (got {st}).")
    sv = to_float(take("NetCDF saving time"), "NetCDF saving time")
    if sv is not None and sv == 0:
        warnings.append("NetCDF saving time is 0 (use -1 to disable field output).")

    # ── Time-series block (conditional) ───────────────────────────────────────
    rp = to_int(take("read points from file?"), "read points from file?")
    if rp == 1:
        check_file(take("POIs file"), "POIs file")
        to_float(take("time-series saving time"), "time-series saving time")
    elif rp not in (0, None):
        errors.append(f"'read points from file?' must be 0 or 1 (got {rp}).")

    # ── Numerics ──────────────────────────────────────────────────────────────
    cfl = to_float(take("CFL"), "CFL")
    if cfl is not None and not (0 < cfl <= 1):
        errors.append(f"CFL must be in (0, 1] (got {cfl}).")
    eps = to_float(take("epsilon_h"), "epsilon_h")
    if eps is not None and eps <= 0:
        warnings.append(f"epsilon_h = {eps} should be a small positive number.")
    to_float(take("WAF threshold"), "WAF threshold")

    rstab = take("stability coefficient")
    if rstab is not None:
        vals = rstab[2]
        bad = [v for v in vals if not _is_float(v)]
        if bad:
            errors.append(f"L{rstab[0]}: stability coefficient must be numeric (got '{rstab[1]}').")
        elif nlev and len(vals) != nlev:
            warnings.append(f"L{rstab[0]}: {len(vals)} stability value(s) for {nlev} level(s) "
                            f"(expected {nlev}).")

    ft = to_int(take("friction type"), "friction type")
    if ft not in (0, 1, 2, None):
        warnings.append(f"Friction type = {ft} is unusual (0 = Manning).")
    man = to_float(take("Manning coefficient"), "Manning coefficient")
    if man is not None and not (0 <= man <= 0.2):
        warnings.append(f"Manning coefficient = {man} outside typical [0, 0.2].")
    to_float(take("maximum water velocity"), "maximum water velocity")
    to_float(take("typical length L"), "typical length L")
    to_float(take("typical height H"), "typical height H")
    # The arrival-time threshold is the last line and is OPTIONAL: some valid
    # files (e.g. the Gulf of Cadiz cases) end at 'typical height' without it.
    to_float(take("arrival-time threshold", optional=True), "arrival-time threshold")

    # ── Trailing content ──────────────────────────────────────────────────────
    if i < len(rows):
        extra = rows[i]
        warnings.append(f"L{extra[0]}: {len(rows) - i} unexpected extra line(s) after the "
                        f"last parameter (first: '{extra[1]}').")
    return errors, warnings


def _is_float(s):
    try:
        float(s); return True
    except ValueError:
        return False


def report(path, **kw):
    """Validate and print a human-readable report."""
    print("=" * 70)
    print(f"Validating: {path}")
    print("=" * 70)
    errors, warnings = validate_parfile(path, **kw)
    if errors:
        print(f"\n  {len(errors)} ERROR(S):")
        for e in errors:    print("   ✗", e)
    if warnings:
        print(f"\n  {len(warnings)} WARNING(S):")
        for w in warnings:  print("   ⚠", w)
    if not errors and not warnings:
        print("\n  ✓ All checks passed — file looks correct.")
    elif not errors:
        print("\n  ✓ No structural errors (warnings are advisory).")
    print()
    return errors, warnings

print("validate_parfile() and report() defined.")

In [ ]:
# ── Validate the reference files (if present); otherwise validate the
# ── single-grid example generated with build_parfile() above ─────────────────
found = [p for p in (PARFILE_1L, PARFILE_ND) if os.path.exists(p)]

if found:
    for p in found:
        report(p)
else:
    print("(reference files not found — validating the generated example instead)\n")
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as tf:
        tf.write(build_parfile(config_single))
        tmp = tf.name
    try:
        report(tmp, check_files=False)
    finally:
        os.unlink(tmp)

---
## 6 — See the validator catch a real mistake

The most common error is **Rule 2**: setting *"read points from file"* = `1` but forgetting the
time-series saving-time line (or vice-versa). That single missing line shifts every numeric
parameter below it by one position — the CFL gets read as the POIs interval, and so on.

The cell below writes a deliberately broken file (the time-series interval removed) and validates
it, so you can see the shift surface as concrete errors.


In [ ]:
# Build a valid file, then break Rule 2: remove the time-series saving-time line
good = build_parfile(config_single)

broken_lines = []
for ln in good.splitlines():
    # drop the "time-series saving time" line while keeping read-points = 1
    if "time-series saving time" in ln:
        continue
    broken_lines.append(ln)
broken = "\n".join(broken_lines) + "\n"

with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as tf:
    tf.write(broken)
    broken_path = tf.name

print("Broken file (time-series line removed):\n")
print(broken)
report(broken_path, check_files=False)
os.unlink(broken_path)

---
## Summary & checklist

You now have a complete picture of the parameter file plus two reusable tools.

| Task | How |
|------|-----|
| Understand a line | Sections 1–3 (annotated tables + the four rules) |
| Build a new file | fill a `config` dict → `build_parfile(config)` → write to `.txt` |
| Check a file | `report("path/to/parameters.txt")` |

**Before you launch, confirm:**

- [ ] Bathymetry / nested grids exist and use the **negative-is-ocean** convention.
- [ ] `number of faults` matches the count of Okada lines.
- [ ] Time-series block present **iff** *read points from file* = `1`.
- [ ] Stability coefficient has **one value per level**.
- [ ] Save flags are 10 long and only `0`/`1`.
- [ ] `validate_parfile` reports **no errors**.

➡ Next steps in this collection:

- **[JN04](JN04_Grid_from_GEBCO.ipynb)** — build the bathymetric grids your
  parameter file references, starting from GEBCO.
- **[JN12](JN12_Format_Conversion.ipynb)** — convert grids in other formats
  (GeoTIFF, Surfer) to the HySEA `.grd` format.
- A forthcoming **processing** notebook will cover launching the validated
  file with `TsunamiHySEA` (directly or via SLURM) — check the `processing/`
  folder of the HySEALab repository for updates.